# GDELT Event Data Analysis

In [ ]:
from pyspark.sql.types import StructType, StructField, FloatType, LongType, StringType, IntegerType
from pyspark.shell import spark

In [ ]:
attrs = []
f = open('./GDELT-EVENTS-ATTRIBUTES.txt')

for line in f:
    tokens = line.split(',')
    if tokens[1].strip() == "INTEGER":
        attrs.append(StructField(tokens[0].strip(), IntegerType(), True))
    elif tokens[1].strip() == "STRING":
        attrs.append(StructField(tokens[0].strip(), StringType(), True))
    elif tokens[1].strip() == "FLOAT":
        attrs.append(StructField(tokens[0].strip(), FloatType(), True))
    
schema = StructType(attrs)
schema

In [ ]:
feats = []
feats.append(StructField('Country_Code', StringType(), True))
feats.append(StructField('Country_Name', StringType(), True))
schema_df_codes = StructType(feats)

In [ ]:
df_country_codes = spark.read.format('CSV').option('sep', ',').schema(schema_df_codes).load('inputs/gdelt/fips-codes.csv')

In [ ]:
data_aids = [('070', 'Provide Aid'),('071','Provide Economic Aid'),
     ('072','Provide Military Aid'),('073','Provide humanitarian aid'),
     ('074','Provide military protection or peacekeeping'),
     ('075','Grant Asylum')]
df_aids = spark.createDataFrame(data_aids,['Event_Code','Type_of_Event'])

In [ ]:
data_deport = [('174','Expel or deport individuals')]
df_deport = spark.createDataFrame(data_deport,['Event_Code','Type_of_Event'])

In [ ]:
data_destrn = [('200','Use massive unconventional force'),
                ('201','Engage in mass expulsion'),
                ('202','Engage in mass killings'),
                ('203','Engage in ethnic cleansing'),
                ('204','Use weapons of mass destruction')]
df_massdestrn = spark.createDataFrame(data_destrn,['Event_Code','Type_of_Event'])

In [ ]:
from pyspark.sql import Column as col
from pyspark.sql.functions import udf
import pyspark.sql.functions as fun

def concat_asc_codes(str1,str2):
    list_codes = []
    list_codes.append(str1)
    list_codes.append(str2)
    list_codes.sort()
    return list_codes[0]+"_"+list_codes[1]

def do_aid_analysis(df,event_code,map_names_event):
    #Filter events required for analysis
    #Group using country codes and event base code
    #joining to country code table to get names of country codes
    aid_maps_1 = df.select('*').where(df.EventRootCode == event_code).groupBy(df.Actor1Geo_CountryCode,
                            df.Actor2Geo_CountryCode,
                            df.EventBaseCode).agg(fun.count('*').alias('no_of_events'),
                                                  fun.sum('GoldsteinScale').alias('TotalGoldsteinScale'),
                                                  fun.sum('NumMentions').alias('TotalNumMentions'),
                                                  fun.sum('NumSources').alias('TotalNumSources'),
                                                  fun.sum('NumArticles').alias('TotalNumArticles'),
                                                  fun.sum('AvgTone').alias('TotalAvgTone')).join(df_country_codes,
                            df.Actor1Geo_CountryCode == df_country_codes.Country_Code)
    aid_maps_1 = aid_maps_1.select(aid_maps_1.Actor1Geo_CountryCode ,
                  aid_maps_1.Country_Name.alias("Actor1Geo_Country_Name"),
                  aid_maps_1.Actor2Geo_CountryCode ,
                  aid_maps_1.EventBaseCode ,
                  aid_maps_1.no_of_events,
                  aid_maps_1.TotalGoldsteinScale,
                  aid_maps_1.TotalNumMentions,
                  aid_maps_1.TotalNumSources,
                  aid_maps_1.TotalNumArticles,
                  aid_maps_1.TotalAvgTone)
    aid_maps_1 = aid_maps_1.join(df_country_codes,aid_maps_1.Actor2Geo_CountryCode == df_country_codes.Country_Code)
    aid_maps = aid_maps_1.select(
                  aid_maps_1.Actor1Geo_CountryCode ,
                  aid_maps_1.Actor1Geo_Country_Name,
                  aid_maps_1.Actor2Geo_CountryCode ,
                  aid_maps_1.Country_Name.alias("Actor2Geo_Country_Name"),
                  aid_maps_1.EventBaseCode ,
                  aid_maps_1.no_of_events,
                  aid_maps_1.TotalGoldsteinScale,
                  aid_maps_1.TotalNumMentions,
                  aid_maps_1.TotalNumSources,
                  aid_maps_1.TotalNumArticles,
                  aid_maps_1.TotalAvgTone)
    #Sorted based on number of events
    sorted_aid_maps = aid_maps.sort(aid_maps.no_of_events.desc())
    #Filtered relations within same country
    diff_actr1_actr2 = sorted_aid_maps.filter(sorted_aid_maps.Actor1Geo_CountryCode != sorted_aid_maps.Actor2Geo_CountryCode)
    diff_actr1_actr2_copy = diff_actr1_actr2
    diff_actr1_actr2_copy = diff_actr1_actr2_copy.select(diff_actr1_actr2_copy.Actor1Geo_CountryCode.alias("CountryCodeA1"),
            diff_actr1_actr2_copy.Actor1Geo_Country_Name.alias("CountryNameA1"),
            diff_actr1_actr2_copy.Actor2Geo_CountryCode.alias("CountryCodeA2"),
            diff_actr1_actr2_copy.Actor2Geo_Country_Name.alias("CountryNameA2"),
            diff_actr1_actr2_copy.EventBaseCode.alias("Event_Base_Code"),
            diff_actr1_actr2_copy.TotalGoldsteinScale.alias("total_goldsteinscale"),
            diff_actr1_actr2_copy.no_of_events.alias("total_number_of_events"),
            diff_actr1_actr2_copy.TotalNumMentions.alias("total_num_mentions"),
            diff_actr1_actr2_copy.TotalNumSources.alias("total_num_sources"),
            diff_actr1_actr2_copy.TotalNumArticles.alias("total_num_articles"),
            diff_actr1_actr2_copy.TotalAvgTone.alias("total_avg_tone")
                    )
    join_df = diff_actr1_actr2_copy.join(diff_actr1_actr2 ,(diff_actr1_actr2_copy.CountryCodeA1==diff_actr1_actr2.Actor2Geo_CountryCode) &
                                               (diff_actr1_actr2_copy.CountryCodeA2==diff_actr1_actr2.Actor1Geo_CountryCode),"outer")
    #Dropping rows with null values
    join_df = join_df.na.drop("all",subset=["Actor1Geo_CountryCode","Actor2Geo_CountryCode"])
    
    #To get strings showing relation btwn two countries
    concat_asc_codes_udf = udf(concat_asc_codes, StringType())
    join_df = join_df.withColumn('Country_Codes_String', concat_asc_codes_udf(join_df.Actor1Geo_CountryCode, join_df.Actor2Geo_CountryCode))

    join_df = join_df.select(
            join_df.Actor1Geo_CountryCode,
            join_df.Actor1Geo_Country_Name,
            join_df.Actor2Geo_CountryCode,
            join_df.Actor2Geo_Country_Name,
            join_df.EventBaseCode,
            join_df.Event_Base_Code,
            join_df.no_of_events,
            join_df.total_number_of_events,
            join_df.Country_Codes_String,
            sum(join_df[cl] for cl in ["total_number_of_events",
                   "no_of_events"]).alias("sum_events"),
            join_df.total_goldsteinscale,
            join_df.total_num_mentions,
            join_df.total_num_sources,
            join_df.total_num_articles,
            join_df.total_avg_tone)
    
    #Dropping duplicates based on relation btwn two countries string
    join_df = join_df.dropDuplicates(['Country_Codes_String']).sort(join_df.sum_events.desc())
    
    #Mapping event codes with event names
    join_df = join_df.join(map_names_event,(join_df.EventBaseCode == map_names_event.Event_Code),'leftouter').select(
                                                          join_df.Country_Codes_String,
                                                          join_df.EventBaseCode,
                                                          join_df.Event_Base_Code,
                                                          map_names_event.Type_of_Event.alias('Type1'),
                                                          join_df.Actor1Geo_Country_Name,
                                                          join_df.Actor2Geo_Country_Name,
                                                          join_df.no_of_events,
                                                          join_df.total_number_of_events,
                                                          join_df.sum_events,
                                                          join_df.total_goldsteinscale,
                                                          join_df.total_num_mentions,
                                                          join_df.total_num_sources,
                                                          join_df.total_num_articles,
                                                          join_df.total_avg_tone
                                                          )
    join_df = join_df.join(map_names_event,(join_df.Event_Base_Code == map_names_event.Event_Code),'leftouter').select(
                                                          join_df.Country_Codes_String,
                                                          join_df.Actor1Geo_Country_Name,
                                                          join_df.Actor2Geo_Country_Name,
                                                          join_df.EventBaseCode,
                                                          join_df.Event_Base_Code,
                                                          join_df.Type1,
                                                          map_names_event.Type_of_Event.alias('Type2'),
                                                          join_df.no_of_events,
                                                          join_df.total_number_of_events,
                                                          join_df.sum_events,
                                                          join_df.total_goldsteinscale,
                                                          join_df.total_num_mentions,
                                                          join_df.total_num_sources,
                                                          join_df.total_num_articles,
                                                          join_df.total_avg_tone
                                                         )
    join_df = join_df.sort(join_df.sum_events.desc())
    return join_df

In [ ]:
def get_state_name(city_state_country,category,country_name):
    tokens = city_state_country.split(',')
    if country_name == 'US':
        if category == 2:
            return tokens[0]
        elif category == 3:
            return tokens[1]
    else :
        if (category == 4) | (category == 5):
            return tokens[1]

def do_state_analysis(df,country_name):
    df_events_with_state = df.select('*').where(df.ActionGeo_CountryCode == country_name)
    df_events_with_state = df_events_with_state.na.drop("all",subset=["ActionGeo_FullName"])
    get_state_name_udf = udf(get_state_name, StringType())
    df_events_with_state = df_events_with_state.withColumn('Action_State', get_state_name_udf(df_events_with_state.ActionGeo_FullName,
                                                                                             df_events_with_state.ActionGeo_Type,
                                                                                             df.ActionGeo_CountryCode))
    df_state_goldstein_events = df_events_with_state.groupBy(df_events_with_state.Action_State,df_events_with_state.EventRootCode).agg(
                                                fun.sum(df_events_with_state.GoldsteinScale).alias("Total_GoldsteinScale"))
    df_state_goldstein_events = df_state_goldstein_events.na.drop("all",subset=["Total_GoldsteinScale"])
    df_state_goldstein_events = df_state_goldstein_events.sort(df_state_goldstein_events.Total_GoldsteinScale.asc())
    return df_state_goldstein_events

# 2015

# AID Analysis

In [ ]:
df_2015 = spark.read.format('CSV').option('sep', '\t').schema(schema).load('inputs/gdelt/2015*')

In [ ]:
df_2015_aid_results = do_aid_analysis(df_2015,'07',df_aids)

In [ ]:
df_2015_aid_results.limit(100).write.csv('spark-outputs/2015-AID-ANALYSIS',header='true')

# DEPORTATION Analysis 

In [ ]:
df_2015_deport_results = do_aid_analysis(df_2015,'17',df_deport)

In [ ]:
df_2015_deport_results = df_2015_deport_results.where(df_2015_deport_results.EventBaseCode == '174')

In [ ]:
df_2015_deport_results.limit(100).write.csv('spark-outputs/2015-DEPORT-ANALYSIS',header='true')

# Top Stories of United States

In [ ]:
top_events_2015_us = df_2015.where((df_2015.Actor1Geo_CountryCode == 'US') |
                          (df_2015.Actor2Geo_CountryCode == 'US')).sort(df_2015.NumMentions.desc())
top_events_2015_us.select('*').limit(50).write.csv('spark-outputs/2015-TOP-STORIES-US',header='true')

# Mass Destruction vs Stability of Countries

In [ ]:
mass_destrn_events = df_2015.where(df_2015.EventRootCode == '20').groupBy(df_2015.ActionGeo_CountryCode).agg(
                        fun.sum(df_2015.GoldsteinScale).alias("Total_GoldsteinScale"),
                        fun.sum(df_2015.AvgTone).alias("Total_AvgTone"),
                        fun.sum(df_2015.NumMentions).alias("Total_NumMentions"))
mass_destrn_events = mass_destrn_events.sort(mass_destrn_events.Total_GoldsteinScale.asc())

In [ ]:
mass_destrn_events = mass_destrn_events.na.drop("all",subset=["ActionGeo_CountryCode"])

In [ ]:
mass_destrn_events = mass_destrn_events.join(df_country_codes,mass_destrn_events.ActionGeo_CountryCode == df_country_codes.Country_Code)

In [ ]:
mass_destrn_events.limit(50).write.csv('spark-outputs/2015-MASS-DESTRUCTION-VS-STABILITY',header='true')

# Pair of Countries involved most in Mass Destruction 

In [ ]:
df_2015_massdestrn_results = do_aid_analysis(df_2015,'20',df_massdestrn)

In [ ]:
df_2015_massdestrn_results = df_2015_massdestrn_results.sort(df_2015_massdestrn_results.total_goldsteinscale.asc())

In [ ]:
df_2015_massdestrn_results = df_2015_massdestrn_results.na.drop("all",subset=["total_goldsteinscale"])

In [ ]:
df_2015_massdestrn_results.limit(100).write.csv('spark-outputs/2015-MASS-DESTRN-ANALYSIS',header='true')

# Which State is effecting stability of US more?

In [ ]:
df_2015_state_stability_country_results = do_state_analysis(df_2015,'US')

In [ ]:
df_2015_state_stability_country_results.limit(100).write.csv('spark-outputs/2015-STATE-EFFECT-COUNTRY-STABILITY-ANALYSIS',header='true')

# Top Stories of India

In [ ]:
top_events_2015_in = df_2015.where((df_2015.ActionGeo_CountryCode == 'IN')).sort(df_2015.NumMentions.desc())
top_events_2015_in.select('*').limit(50).write.csv('spark-outputs/2015-TOP-STORIES-IN',header='true')

# Which State is effecting stability of India more?

In [ ]:
df_2015_state_stability_country_results = do_state_analysis(df_2015,'IN')

In [ ]:
df_2015_state_stability_country_results.limit(100).write.csv('spark-outputs/2015-STATE-EFFECT-COUNTRY-STABILITY-INDIA',header='true')

# 2016

# AID Analysis

In [ ]:
df_2016 = spark.read.format('CSV').option('sep', '\t').schema(schema).load('inputs/gdelt/2016*')

In [ ]:
df_2016_aid_results = do_aid_analysis(df_2016,'07',df_aids)

In [ ]:
df_2016_aid_results.limit(100).write.csv('spark-outputs/2016-AID-ANALYSIS',header='true')

# DEPORTATION Analysis

In [ ]:
df_2016_deport_results = do_aid_analysis(df_2016,'17',df_deport)

In [ ]:
df_2016_deport_results = df_2016_deport_results.where(df_2016_deport_results.EventBaseCode == '174')

In [ ]:
df_2016_deport_results.limit(100).write.csv('spark-outputs/2016-DEPORT-ANALYSIS',header='true')

# Top Stories of United States

In [ ]:
top_events_2016_us = df_2016.where((df_2016.Actor1Geo_CountryCode == 'US') |
                          (df_2016.Actor2Geo_CountryCode == 'US')).sort(df_2016.NumMentions.desc())
top_events_2016_us.select('*').limit(50).write.csv('spark-outputs/2016-TOP-STORIES-US',header='true')

# Mass Destruction vs Stability of Countries

In [ ]:
mass_destrn_events_2016 = df_2016.where(df_2016.EventRootCode == '20').groupBy(df_2016.ActionGeo_CountryCode).agg(
                        fun.sum(df_2016.GoldsteinScale).alias("Total_GoldsteinScale"),
                        fun.sum(df_2016.AvgTone).alias("Total_AvgTone"),
                        fun.sum(df_2016.NumMentions).alias("Total_NumMentions"))
mass_destrn_events_2016 = mass_destrn_events_2016.sort(mass_destrn_events_2016.Total_GoldsteinScale.asc())

In [ ]:
mass_destrn_events_2016 = mass_destrn_events_2016.na.drop("all",subset=["ActionGeo_CountryCode"])

In [ ]:
mass_destrn_events_2016 = mass_destrn_events_2016.join(df_country_codes,mass_destrn_events_2016.ActionGeo_CountryCode == df_country_codes.Country_Code)

In [ ]:
mass_destrn_events_2016.limit(50).write.csv('spark-outputs/2016-MASS-DESTRUCTION-VS-STABILITY',header='true')

# Pair of Countries involved most in Mass Destruction

In [ ]:
df_2016_massdestrn_results = do_aid_analysis(df_2016,'20',df_massdestrn)

In [ ]:
df_2016_massdestrn_results = df_2016_massdestrn_results.sort(df_2016_massdestrn_results.total_goldsteinscale.asc())

In [ ]:
df_2016_massdestrn_results = df_2016_massdestrn_results.na.drop("all",subset=["total_goldsteinscale"])

In [ ]:
df_2016_massdestrn_results.limit(100).write.csv('spark-outputs/2016-MASS-DESTRN-ANALYSIS',header='true')

# Which state is effecting stability of US more?

In [ ]:
df_2016_state_stability_country_results = do_state_analysis(df_2016,'US')

In [ ]:
df_2016_state_stability_country_results.limit(100).write.csv('spark-outputs/2016-STATE-EFFECT-COUNTRY-STABILITY-ANALYSIS',header='true')

# Top Stories of India

In [ ]:
top_events_2016_in = df_2016.where((df_2016.ActionGeo_CountryCode == 'IN')).sort(df_2016.NumMentions.desc())
top_events_2016_in.select('*').limit(50).write.csv('spark-outputs/2016-TOP-STORIES-IN',header='true')

# Which State is effecting stability of India more?

In [ ]:
df_2016_state_stability_country_results_in = do_state_analysis(df_2016,'IN')

In [ ]:
df_2016_state_stability_country_results_in.limit(100).write.csv('spark-outputs/2016-STATE-EFFECT-COUNTRY-STABILITY-INDIA',header='true')

# Scrapbook

In [ ]:
df_country_codes.take(7)

In [ ]:
df.take(1)

In [ ]:
df.count()

In [ ]:
df_2015.take(1)

In [ ]:
df_2015.count()

In [ ]:
df_2015.columns

# Mass Destruction - Israel Analysis

In [ ]:
mass_destrn_events = df_2015.where((df_2015.EventRootCode == '20') & (df_2015.ActionGeo_CountryCode == 'IS')).select('*')
mass_destrn_events = mass_destrn_events.sort(mass_destrn_events.GoldsteinScale.asc())
mass_destrn_events.take(5)

In [ ]:
top_events_2015_india = df_2015.where((df_2015.Actor1CountryCode == 'IND') |
                          (df_2015.Actor2CountryCode == 'IND')).sort(df_2015.NumMentions.desc())
top_events_2015_india.select(top_events_2015_india.Actor1Name,
                             top_events_2015_india.Actor2Name,
                             top_events_2015_india.NumMentions,
                             top_events_2015_india.NumSources,
                             top_events_2015_india.AvgTone,
                             top_events_2015_india.SOURCEURL).show(n=5)

In [ ]:
top_events_2016_india = df.where((df.Actor1CountryCode == 'IND') |
              (df.Actor2CountryCode == 'IND')).select(*df.columns).sort(df.NumMentions.desc())
top_events_2016_india.select(top_events_2016_india.Actor1Name,
                             top_events_2016_india.Actor2Name,
                             top_events_2016_india.NumMentions,
                             top_events_2016_india.NumSources,
                             top_events_2016_india.AvgTone,
                             top_events_2016_india.SOURCEURL).show(n=5)

In [ ]:
top_events_2015_india.select(top_events_2015_india.Actor1Name,
                             top_events_2015_india.Actor2Name,
                             top_events_2015_india.NumMentions,
                             top_events_2015_india.NumSources,
                             top_events_2015_india.AvgTone,
                             top_events_2015_india.SOURCEURL).limit(5).write.csv('top_events_2015_india.csv')

In [ ]:
top_events_2016_india.select(top_events_2016_india.Actor1Name,
                             top_events_2016_india.Actor2Name,
                             top_events_2016_india.NumMentions,
                             top_events_2016_india.NumSources,
                             top_events_2016_india.AvgTone,
                             top_events_2016_india.SOURCEURL).limit(5).write.csv('top_events_2016_india.csv')

In [ ]:
top_events_2015_india.sample(False,0.1).select('*').limit(100).write.csv('top_events_2015_sample.csv')

In [ ]:
df_2015.sample(False,0.05).select('*').where(df_2015.EventRootCode == '07').limit(100).write.csv('2015_AID')

In [ ]:
import pyspark.sql.functions as fun
df_2015.sample(False,0.05).select('*').where(df_2015.EventRootCode == '07').groupBy(df_2015.Actor2Geo_CountryCode).agg(fun.count(df_2015.Actor2Geo_CountryCode).alias('no_of_times_aid_received')).join(df_country_codes,
                        df_2015.Actor2Geo_CountryCode == df_country_codes.Nato_Country_Code).limit(100).write.csv('2015_AID_RECEIVED_COUNTRIES')

In [ ]:
df_2015.select('*').where(df_2015.EventCode == '174').take(5)

In [ ]:
df_2015.select('*').where(df_2015.EventRootCode == '07').groupBy(df_2015.Actor2Geo_CountryCode).agg(fun.count(df_2015.Actor2Geo_CountryCode).alias('no_of_times_aid_received')).join(df_country_codes,
                        df_2015.Actor2Geo_CountryCode == df_country_codes.Nato_Country_Code).limit(100).write.csv('2015_AID_RECEIVED_COUNTRIES_FULL_DATA')

In [ ]:
df_2015.select('*').where(df_2015.EventRootCode == '07').groupBy(df_2015.Actor1Geo_CountryCode).agg(fun.count(df_2015.Actor1Geo_CountryCode).alias('no_of_times_aid_received')).join(df_country_codes,
                        df_2015.Actor1Geo_CountryCode == df_country_codes.Nato_Country_Code).limit(100).write.csv('2015_AID_RECEIVED_COUNTRIES_FULL_DATA_ACTOR1')

In [ ]:
mass_destrn_events = df_2015.where(df_2015.EventRootCode == '20')
mass_destrn_events.select('*').take(5)